In [1]:
# ics_hybrid_phase1_phase2_swat_FAST.py
# ------------------------------------------------------------
# Phase-1 + Phase-2 Single-file (SWaT Kaggle Dataset)
# Hybrid: FAST SARIMA (downsampled) + LSTM Autoencoder + OR Fusion
#
# Fixes:
# - KaggleHub empty FILE_PATH -> dataset_download + merged.csv
# - Colab/Jupyter argparse -f kernel.json -> parse_known_args
# - SARIMA forecast length mismatch -> rolling + interpolation
# - SARIMA too slow -> downsample + smaller fit length
# ------------------------------------------------------------
# Install:
#   pip install "numpy<2" pandas scikit-learn statsmodels tensorflow joblib matplotlib kagglehub streamlit
# ------------------------------------------------------------

import os, glob, json, time, pickle, argparse
from collections import deque

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import kagglehub
from kagglehub import KaggleDatasetAdapter

import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from statsmodels.tsa.statespace.sarimax import SARIMAX

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

try:
    import streamlit as st
    STREAMLIT_OK = True
except Exception:
    STREAMLIT_OK = False


# =========================
# CONFIG (FAST)
# =========================
KAGGLE_HANDLE = "vishala28/swat-dataset-secure-water-treatment-system"
FILE_PATH = ""  # keep empty -> auto pick merged.csv

OUT_DIR = "ics_swat_outputs"
MODEL_DIR = "ics_swat_models"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

MAX_ROWS = 120000          # ✅ reduce from 200k
SARIMA_TRAIN_MAX = 6000    # ✅ reduce from 20000
SARIMA_DOWNSAMPLE = 5      # ✅ fit SARIMA on every 5th point -> much faster

TRAIN_FRAC = 0.70
VAL_FRAC = 0.15

SEQ_LEN = 30
PCA_COMPONENTS_FOR_LSTM = 4

SARIMA_ORDER = (2, 1, 2)
SEASONAL_ORDER = (1, 0, 1, 24)

SIGMA_K = 3.0

# Threshold/fusion tuning on validation split
TUNE_QUANTILES = [0.99, 0.995, 0.997]
FUSION_MODES = ["or", "and"]
PERSISTENCE_STEPS = [1, 2, 3]


# =========================
# HELPERS
# =========================
def infer_columns(df: pd.DataFrame):
    cols = list(df.columns)
    ts_candidates = [c for c in cols if "time" in c.lower() or "date" in c.lower()]
    timestamp_col = ts_candidates[0] if ts_candidates else None

    label_candidates = []
    for c in cols:
        cl = c.lower().strip()
        if cl in ["label", "attack", "is_attack", "anomaly", "normal/attack", "normal_attack", "attack_label"]:
            label_candidates.append(c)
    label_col = label_candidates[0] if label_candidates else None

    sensor_cols = []
    for c in cols:
        if c == timestamp_col or c == label_col:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            sensor_cols.append(c)

    if not sensor_cols:
        for c in cols:
            if c == timestamp_col or c == label_col:
                continue
            df[c] = pd.to_numeric(df[c], errors="coerce")
            if df[c].notna().sum() > 0:
                sensor_cols.append(c)

    return timestamp_col, label_col, sensor_cols


def normalize_attack_label(series: pd.Series):
    if series is None:
        return None
    if pd.api.types.is_numeric_dtype(series):
        return (series.astype(float).fillna(0.0) > 0.5).to_numpy(dtype=bool)
    s = series.astype(str).str.strip().str.lower()
    out = np.zeros(len(s), dtype=bool)
    for i, v in enumerate(s):
        out[i] = ("attack" in v) or (v == "1") or (v == "true") or (v == "a")
    return out


def safe_time_sort(df: pd.DataFrame, timestamp_col: str):
    if timestamp_col is None:
        df = df.copy()
        df["__t"] = np.arange(len(df))
        return df, "__t"
    df = df.copy()
    df[timestamp_col] = pd.to_datetime(df[timestamp_col], errors="coerce", dayfirst=True)
    if pd.api.types.is_datetime64_any_dtype(df[timestamp_col]):
        df = df.sort_values(timestamp_col).reset_index(drop=True)
    return df, timestamp_col


def split_indices(n: int, train_frac: float, val_frac: float):
    n_train = int(n * train_frac)
    n_val = int(n * val_frac)
    train_idx = np.arange(0, n_train)
    val_idx = np.arange(n_train, n_train + n_val)
    test_idx = np.arange(n_train + n_val, n)
    return train_idx, val_idx, test_idx


def make_sequences(X2d: np.ndarray, seq_len: int):
    X2d = np.asarray(X2d, dtype=float)
    Xs, ends = [], []
    for end in range(seq_len - 1, len(X2d)):
        start = end - seq_len + 1
        w = X2d[start:end + 1]
        if np.any(np.isnan(w)):
            continue
        Xs.append(w)
        ends.append(end)
    return np.array(Xs, dtype=float), np.array(ends, dtype=int)


def build_lstm_autoencoder(seq_len: int, n_features: int):
    inp = layers.Input(shape=(seq_len, n_features))
    x = layers.LSTM(64, activation="tanh")(inp)
    x = layers.RepeatVector(seq_len)(x)
    x = layers.LSTM(64, activation="tanh", return_sequences=True)(x)
    out = layers.TimeDistributed(layers.Dense(n_features))(x)
    m = models.Model(inp, out)
    m.compile(optimizer="adam", loss="mse")
    return m


def seq_mse(X_true: np.ndarray, X_pred: np.ndarray):
    return np.mean(np.square(X_pred - X_true), axis=(1, 2))


def metrics_report(y_true: np.ndarray, y_pred: np.ndarray):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[False, True]).ravel()
    fpr = fp / (fp + tn + 1e-9)
    return {"accuracy": float(acc), "precision": float(prec), "recall": float(rec),
            "f1": float(f1), "false_positive_rate": float(fpr),
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}


def rolling_sarima_forecast(fitted_res, y_stream):
    y_stream = np.asarray(y_stream, dtype=float)
    if len(y_stream) == 0:
        return np.array([], dtype=float)

    # Fast path for long streams; fallback keeps compatibility.
    try:
        return np.asarray(fitted_res.forecast(steps=len(y_stream)), dtype=float)
    except Exception:
        preds = []
        res_state = fitted_res
        for v in y_stream:
            p = float(res_state.forecast(steps=1)[0])
            preds.append(p)
            res_state = res_state.append(endog=[float(v)], refit=False)
        return np.array(preds, dtype=float)


def apply_persistence(flags: np.ndarray, min_steps: int):
    flags = np.asarray(flags, dtype=bool)
    if min_steps <= 1:
        return flags
    out = np.zeros_like(flags, dtype=bool)
    run = 0
    for i, v in enumerate(flags):
        run = (run + 1) if v else 0
        if run >= min_steps:
            out[i] = True
    return out


def fuse_flags(sarima_flag: np.ndarray, lstm_flag: np.ndarray, mode: str, persistence_steps: int):
    if mode == "and":
        base = np.asarray(sarima_flag, dtype=bool) & np.asarray(lstm_flag, dtype=bool)
    else:
        base = np.asarray(sarima_flag, dtype=bool) | np.asarray(lstm_flag, dtype=bool)
    return apply_persistence(base, persistence_steps)


def downsample(arr, k):
    if k is None or k <= 1:
        return np.asarray(arr, dtype=float)
    return np.asarray(arr, dtype=float)[::k]


def load_swat_dataframe(nrows=None) -> pd.DataFrame:
    if FILE_PATH and os.path.splitext(FILE_PATH)[1]:
        if hasattr(kagglehub, "dataset_load"):
            return kagglehub.dataset_load(
                KaggleDatasetAdapter.PANDAS,
                handle=KAGGLE_HANDLE,
                path=FILE_PATH,
                pandas_kwargs={"nrows": nrows} if nrows else None,
            )
        return kagglehub.load_dataset(
            KaggleDatasetAdapter.PANDAS,
            KAGGLE_HANDLE,
            FILE_PATH,
            pandas_kwargs={"nrows": nrows} if nrows else None,
        )

    root = kagglehub.dataset_download(KAGGLE_HANDLE)
    csvs = glob.glob(os.path.join(root, "**", "*.csv"), recursive=True)
    if not csvs:
        raise RuntimeError("No CSV files found in SWaT dataset download.")
    merged = [p for p in csvs if os.path.basename(p).lower() == "merged.csv"]
    if merged:
        csv_path = merged[0]
    else:
        csv_path = sorted(csvs, key=lambda p: os.path.getsize(p), reverse=True)[0]
    return pd.read_csv(csv_path, nrows=nrows)


# =========================
# PHASE-1: TRAIN + EVAL
# =========================
def train_phase1():
    print("📥 Loading SWaT from KaggleHub...")
    df = load_swat_dataframe(nrows=MAX_ROWS if MAX_ROWS else None)
    print("✅ Loaded rows:", len(df))

    ts_col, label_col, sensor_cols = infer_columns(df)
    df, ts_col = safe_time_sort(df, ts_col)

    attack = None
    if label_col is not None and label_col in df.columns:
        attack = normalize_attack_label(df[label_col])

    print("Timestamp:", ts_col, "| Label:", label_col, "| Sensors:", len(sensor_cols))

    X = df[sensor_cols].apply(pd.to_numeric, errors="coerce")
    X = X.replace([np.inf, -np.inf], np.nan).interpolate(limit_direction="both").ffill().bfill()

    n = len(X)
    train_idx, val_idx, test_idx = split_indices(n, TRAIN_FRAC, VAL_FRAC)

    train_end = train_idx[-1] + 1
    if attack is not None and np.any(attack):
        first_attack = int(np.argmax(attack))
        if first_attack > (SEQ_LEN * 5):
            train_end = min(train_end, first_attack)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X.iloc[:train_end].values)
    X_all_scaled = scaler.transform(X.values)

    n_comp = min(PCA_COMPONENTS_FOR_LSTM, X_all_scaled.shape[1])
    pca = PCA(n_components=n_comp, random_state=42)
    Z_train = pca.fit_transform(X_train_scaled)
    Z_all = pca.transform(X_all_scaled)

    # ========== SARIMA FAST ==========
    y_train = Z_train[:, 0].astype(float)
    if SARIMA_TRAIN_MAX and len(y_train) > SARIMA_TRAIN_MAX:
        y_train = y_train[-SARIMA_TRAIN_MAX:]
    y_fit = downsample(y_train, SARIMA_DOWNSAMPLE)

    print(f"🧠 Fitting SARIMA on {len(y_fit)} points (downsample={SARIMA_DOWNSAMPLE})...")
    sarima = SARIMAX(
        y_fit,
        order=SARIMA_ORDER,
        seasonal_order=SEASONAL_ORDER,
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    sarima_res = sarima.fit(disp=False)

    pred_in = sarima_res.get_prediction(start=0, end=len(y_fit)-1).predicted_mean
    resid_fit = y_fit - np.asarray(pred_in, dtype=float)

    stream_start = train_end
    y_stream_full = Z_all[stream_start:, 0].astype(float)
    y_stream_ds = downsample(y_stream_full, SARIMA_DOWNSAMPLE)

    pred_ds = rolling_sarima_forecast(sarima_res, y_stream_ds)
    x_full = np.arange(len(y_stream_full))
    x_ds = np.arange(0, len(y_stream_full), SARIMA_DOWNSAMPLE)
    pred_full = np.interp(x_full, x_ds, pred_ds)

    resid_stream = y_stream_full - pred_full
    sarima_score = np.abs(resid_stream)
    stream_len = len(y_stream_full)

    # ========== LSTM AE ==========
    print(f"🧠 Training LSTM AE ({SEQ_LEN}, {n_comp}) ...")
    X_seq_train, _ = make_sequences(Z_train, SEQ_LEN)
    if len(X_seq_train) < 50:
        raise RuntimeError("Not enough sequences for LSTM training. Reduce SEQ_LEN or increase MAX_ROWS.")

    ae = build_lstm_autoencoder(SEQ_LEN, n_comp)
    es = callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

    ae.fit(
        X_seq_train, X_seq_train,
        epochs=10,
        batch_size=128,
        validation_split=0.2,
        callbacks=[es],
        verbose=1
    )

    train_pred = ae.predict(X_seq_train, verbose=0)
    train_mse = seq_mse(X_seq_train, train_pred)

    Z_stream = Z_all[stream_start:, :]
    X_seq_stream, stream_end_idx = make_sequences(Z_stream, SEQ_LEN)
    stream_pred = ae.predict(X_seq_stream, verbose=0)
    stream_mse = seq_mse(X_seq_stream, stream_pred)

    lstm_score = np.full(stream_len, np.nan, dtype=float)
    for k, e in enumerate(stream_end_idx):
        lstm_score[e] = float(stream_mse[k])

    # =========================
    # Validation-driven tuning
    # =========================
    val_len = min(len(val_idx), stream_len)
    val_slice = slice(0, val_len)

    if val_len > max(SEQ_LEN, 50):
        sarima_val = sarima_score[val_slice]
        lstm_val = lstm_score[val_slice]

        y_val = None
        if attack is not None:
            y_val = attack[stream_start:stream_start + val_len].astype(bool)

        best = None
        for q_s in TUNE_QUANTILES:
            for q_l in TUNE_QUANTILES:
                sar_thr = float(np.quantile(sarima_val, q_s))
                finite_lstm_val = lstm_val[np.isfinite(lstm_val)]
                if len(finite_lstm_val) == 0:
                    continue
                lst_thr = float(np.quantile(finite_lstm_val, q_l))

                sar_flag_val = sarima_val > sar_thr
                lst_flag_val = np.zeros(val_len, dtype=bool)
                valid_val = np.isfinite(lstm_val)
                lst_flag_val[valid_val] = lstm_val[valid_val] > lst_thr

                for mode in FUSION_MODES:
                    for p_steps in PERSISTENCE_STEPS:
                        comb_val = fuse_flags(sar_flag_val, lst_flag_val, mode, p_steps)

                        if y_val is not None and np.any(y_val) and np.any(~y_val):
                            score = f1_score(y_val, comb_val, zero_division=0)
                        else:
                            score = accuracy_score(np.zeros(val_len, dtype=bool), comb_val)

                        alert_rate = float(np.mean(comb_val.astype(float)))
                        candidate = {
                            "score": float(score),
                            "alert_rate": alert_rate,
                            "sarima_thr": sar_thr,
                            "lstm_thr": lst_thr,
                            "fusion_mode": mode,
                            "persistence_steps": int(p_steps),
                        }
                        if best is None or (candidate["score"] > best["score"]) or (
                            np.isclose(candidate["score"], best["score"]) and candidate["alert_rate"] < best["alert_rate"]
                        ):
                            best = candidate

        if best is not None:
            sarima_thr = best["sarima_thr"]
            lstm_thr = best["lstm_thr"]
            fusion_mode = best["fusion_mode"]
            persistence_steps = best["persistence_steps"]
        else:
            sarima_thr = float(np.mean(np.abs(resid_fit)) + SIGMA_K * np.std(np.abs(resid_fit)))
            lstm_thr = float(np.mean(train_mse) + SIGMA_K * np.std(train_mse))
            fusion_mode = "or"
            persistence_steps = 1
    else:
        sarima_thr = float(np.mean(np.abs(resid_fit)) + SIGMA_K * np.std(np.abs(resid_fit)))
        lstm_thr = float(np.mean(train_mse) + SIGMA_K * np.std(train_mse))
        fusion_mode = "or"
        persistence_steps = 1

    sarima_flag = sarima_score > sarima_thr
    lstm_flag = np.zeros(stream_len, dtype=bool)
    valid = ~np.isnan(lstm_score)
    lstm_flag[valid] = lstm_score[valid] > lstm_thr

    combined_flag = fuse_flags(sarima_flag, lstm_flag, fusion_mode, persistence_steps)

    # Save models
    joblib.dump(scaler, os.path.join(MODEL_DIR, "scaler.joblib"))
    joblib.dump(pca, os.path.join(MODEL_DIR, "pca.joblib"))
    with open(os.path.join(MODEL_DIR, "sarima.pkl"), "wb") as f:
        pickle.dump(sarima_res, f)
    ae.save(os.path.join(MODEL_DIR, "lstm_autoencoder.keras"))

    cfg = {
        "kaggle_handle": KAGGLE_HANDLE,
        "timestamp_col": ts_col,
        "label_col": label_col,
        "sensor_cols": sensor_cols,
        "train_end": int(train_end),
        "seq_len": int(SEQ_LEN),
        "pca_components": int(n_comp),
        "sarima_order": list(SARIMA_ORDER),
        "seasonal_order": list(SEASONAL_ORDER),
        "sarima_threshold": float(sarima_thr),
        "lstm_threshold": float(lstm_thr),
        "sigma_k": float(SIGMA_K),
        "tune_quantiles": [float(x) for x in TUNE_QUANTILES],
        "fusion_modes": list(FUSION_MODES),
        "persistence_candidates": [int(x) for x in PERSISTENCE_STEPS],
        "selected_fusion_mode": str(fusion_mode),
        "selected_persistence_steps": int(persistence_steps),
        "max_rows": (None if MAX_ROWS is None else int(MAX_ROWS)),
        "sarima_train_max": int(SARIMA_TRAIN_MAX),
        "sarima_downsample": int(SARIMA_DOWNSAMPLE),
    }
    with open(os.path.join(MODEL_DIR, "config.json"), "w") as f:
        json.dump(cfg, f, indent=2)

    out = pd.DataFrame({
        "global_index": np.arange(stream_start, n),
        "sarima_score": sarima_score,
        "sarima_alert": sarima_flag.astype(int),
        "lstm_score": lstm_score,
        "lstm_alert": lstm_flag.astype(int),
        "combined_alert": combined_flag.astype(int),
    })

    if ts_col in df.columns:
        out["timestamp"] = df[ts_col].iloc[stream_start:].reset_index(drop=True)
    if attack is not None:
        out["attack_gt"] = attack[stream_start:].astype(int)

    results_path = os.path.join(OUT_DIR, "detection_results.csv")
    out.to_csv(results_path, index=False)

    alerts_path = os.path.join(OUT_DIR, "alerts.csv")
    out[out["combined_alert"] == 1].to_csv(alerts_path, index=False)

    print("📄 Saved:", results_path)
    print("📄 Saved:", alerts_path)

    if attack is not None:
        warmup = (SEQ_LEN - 1)
        mask = np.arange(len(out)) >= warmup
        y_true = out.loc[mask, "attack_gt"].astype(int).to_numpy().astype(bool)
        y_pred = out.loc[mask, "combined_alert"].astype(int).to_numpy().astype(bool)
        report = metrics_report(y_true, y_pred)
        with open(os.path.join(OUT_DIR, "metrics.json"), "w") as f:
            json.dump(report, f, indent=2)
        print("📊 Metrics:", report)

    print("\n✅ TRAIN DONE")
    print("Models folder:", MODEL_DIR)
    print("Outputs folder:", OUT_DIR)


# =========================
# PHASE-2: REAL-TIME SIMULATION
# =========================
def simulate_phase2(simulate_steps: int = 20000, sleep_s: float = 0.0):
    cfg_path = os.path.join(MODEL_DIR, "config.json")
    if not os.path.exists(cfg_path):
        raise RuntimeError("No trained model found. Run train_phase1() first.")

    cfg = json.load(open(cfg_path, "r"))
    seq_len = int(cfg["seq_len"])
    train_end = int(cfg["train_end"])
    sarima_thr = float(cfg["sarima_threshold"])
    lstm_thr = float(cfg["lstm_threshold"])
    k = int(cfg.get("sarima_downsample", 5))
    fusion_mode = str(cfg.get("selected_fusion_mode", "or"))
    persistence_steps = int(cfg.get("selected_persistence_steps", 1))

    scaler = joblib.load(os.path.join(MODEL_DIR, "scaler.joblib"))
    pca = joblib.load(os.path.join(MODEL_DIR, "pca.joblib"))
    sarima_res = pickle.load(open(os.path.join(MODEL_DIR, "sarima.pkl"), "rb"))
    ae = tf.keras.models.load_model(os.path.join(MODEL_DIR, "lstm_autoencoder.keras"))

    df = load_swat_dataframe(nrows=MAX_ROWS if MAX_ROWS else None)
    ts_col, label_col, sensor_cols = infer_columns(df)
    df, ts_col = safe_time_sort(df, ts_col)

    attack = None
    if label_col is not None and label_col in df.columns:
        attack = normalize_attack_label(df[label_col])

    X = df[sensor_cols].apply(pd.to_numeric, errors="coerce")
    X = X.replace([np.inf, -np.inf], np.nan).interpolate(limit_direction="both").ffill().bfill()

    Z_all = pca.transform(scaler.transform(X.values))
    stream_len = len(df) - train_end
    steps = min(simulate_steps, stream_len)

    stream_start = train_end
    y_stream_full = Z_all[stream_start:stream_start + steps, 0].astype(float)
    y_stream_ds = y_stream_full[::k]
    pred_ds = rolling_sarima_forecast(sarima_res, y_stream_ds)

    x_full = np.arange(len(y_stream_full))
    x_ds = np.arange(0, len(y_stream_full), k)
    pred_full = np.interp(x_full, x_ds, pred_ds)

    sarima_score = np.abs(y_stream_full - pred_full)
    sarima_flag = sarima_score > sarima_thr

    buf = deque(maxlen=seq_len)
    rows = []
    raw_combined = []

    for i in range(steps):
        buf.append(Z_all[stream_start + i, :])

        lstm_score = np.nan
        lstm_flag = False
        if len(buf) == seq_len:
            X_seq = np.array(buf, dtype=float).reshape(1, seq_len, Z_all.shape[1])
            pred_seq = ae.predict(X_seq, verbose=0)
            lstm_score = float(seq_mse(X_seq, pred_seq)[0])
            lstm_flag = lstm_score > lstm_thr

        if fusion_mode == "and":
            raw = bool(sarima_flag[i] and lstm_flag)
        else:
            raw = bool(sarima_flag[i] or lstm_flag)
        raw_combined.append(raw)

        gt = None if attack is None else bool(attack[stream_start + i])

        rows.append({
            "step": i,
            "global_index": int(stream_start + i),
            "timestamp": (str(df[ts_col].iloc[stream_start + i]) if ts_col in df.columns else int(stream_start + i)),
            "sarima_score": float(sarima_score[i]),
            "sarima_alert": int(sarima_flag[i]),
            "lstm_score": (None if np.isnan(lstm_score) else float(lstm_score)),
            "lstm_alert": int(lstm_flag),
            "combined_alert": 0,
            "ground_truth_attack": (None if gt is None else int(gt)),
        })

        if sleep_s > 0:
            time.sleep(sleep_s)

    final_combined = fuse_flags(np.asarray(raw_combined, dtype=bool), np.asarray(raw_combined, dtype=bool), "or", persistence_steps)
    for i in range(len(rows)):
        rows[i]["combined_alert"] = int(final_combined[i])

    log_df = pd.DataFrame(rows)
    log_path = os.path.join(OUT_DIR, "realtime_log.csv")
    log_df.to_csv(log_path, index=False)
    print("📄 Saved:", log_path)
    print("\n✅ SIMULATION DONE")


# =========================
# DASHBOARD (OPTIONAL)
# =========================
def run_dashboard():
    if not STREAMLIT_OK:
        raise RuntimeError("Install streamlit: pip install streamlit")

    st.title("ICS Hybrid Detection Dashboard (SWaT FAST)")
    st.caption("FAST SARIMA (downsampled) + LSTM AE + OR Fusion")

    if st.button("Train"):
        train_phase1()

    steps = st.number_input("Simulate steps", 1000, 200000, 20000, 1000)
    if st.button("Simulate"):
        simulate_phase2(simulate_steps=int(steps), sleep_s=0.0)

    path = os.path.join(OUT_DIR, "detection_results.csv")
    if os.path.exists(path):
        df = pd.read_csv(path)
        st.dataframe(df.tail(50))
        st.line_chart(df[["sarima_score"]].fillna(0))
        st.line_chart(df[["combined_alert"]])


# =========================
# MAIN
# =========================
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--mode", type=str, default="train", choices=["train", "simulate", "dashboard"])
    parser.add_argument("--simulate_steps", type=int, default=20000)
    parser.add_argument("--sleep", type=float, default=0.0)

    args, _ = parser.parse_known_args()

    if args.mode == "train":
        train_phase1()
    elif args.mode == "simulate":
        simulate_phase2(simulate_steps=args.simulate_steps, sleep_s=args.sleep)
    else:
        run_dashboard()

if __name__ == "__main__":
    main()


C:\Users\S S U S\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📥 Loading SWaT from KaggleHub...


✅ Loaded rows: 120000


Timestamp:  Timestamp | Label: Normal/Attack | Sensors: 51


🧠 Fitting SARIMA on 1200 points (downsample=5)...


...

522/525 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - loss: 0.0781

523/525 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - loss: 0.0781

524/525 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - loss: 0.0781

525/525 ━━━━━━━━━━━━━━━━━━━━ 30s 56ms/step - loss: 0.0772 - val_loss: 1.2546


📄 Saved: ics_swat_outputs\detection_results.csv
📄 Saved: ics_swat_outputs\alerts.csv
📊 Metrics: {'accuracy': 0.9994439965527786, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'false_positive_rate': 0.0005560034472213574, 'tn': 35951, 'fp': 20, 'fn': 0, 'tp': 0}

✅ TRAIN DONE
Models folder: ics_swat_models
Outputs folder: ics_swat_outputs
